**0. Import des bibliothèques**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**1. Import des données**

In [3]:
grants = pd.read_csv("grants.csv", sep=";", encoding="latin1")
nonprofits = pd.read_csv("non-profits_final.csv", sep=";")
quality = pd.read_csv("nonprofit_quality.csv", sep=";")

print("Données importées avec succès.")

print(f"Grants : {grants.shape[0]:,} lignes et {grants.shape[1]} colonnes")
print(f"Nonprofits : {nonprofits.shape[0]:,} lignes et {nonprofits.shape[1]} colonnes")
print(f"Quality : {quality.shape[0]:,} lignes et {quality.shape[1]} colonnes")

C:\Users\PC\AppData\Local\Temp\ipykernel_18972\1683738793.py:2: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  nonprofits = pd.read_csv("non-profits_final.csv", sep=";")


Données importées avec succès.
Grants : 75,337 lignes et 22 colonnes
Nonprofits : 240,585 lignes et 33 colonnes
Quality : 240,585 lignes et 10 colonnes


C:\Users\PC\AppData\Local\Temp\ipykernel_18972\1683738793.py:3: DtypeWarning: Columns (1,5,9) have mixed types. Specify dtype option on import or set low_memory=False.
  quality = pd.read_csv("nonprofit_quality.csv", sep=";")


**2. Exploration initiale**

    * Quelles colonnes avons-nous ?
    * Quels types de données ?
    * Y a-t-il des valeurs manquantes ?
    * Quels sont les premiers exemples de données ?

In [7]:
# Afficher les informations sur les DataFrames
print("Dataset Grants")
grants.info()

print("\n" + "-" * 60 + "\n")

print("Dataset Nonprofits")
nonprofits.info()

print("\n" + "-" * 60 + "\n")

print("Dataset Quality")
quality.info()

Dataset Grants
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75337 entries, 0 to 75336
Data columns (total 22 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Unnamed: 0                            75337 non-null  int64  
 1   opportunity_id                        75337 non-null  int64  
 2   opportunity_title                     75337 non-null  object 
 3   opportunity_number                    69359 non-null  object 
 4   opportunity_category                  69359 non-null  object 
 5   funding_instrument_type               69359 non-null  object 
 6   category_of_funding_activity          69359 non-null  object 
 7   cfda_numbers                          68615 non-null  float64
 8   eligible_applicants                   69359 non-null  object 
 9   eligible_applicants_type              69359 non-null  object 
 10  agency_code                           69323 non-null  object 
 11  

**3 : Audit de qualité :**

    * Vérifier les valeurs manquantes
    * Identifier les anomalies
    * Détecter les doublons
    * Analyser les types de données aberrants

In [9]:
print("=" * 60)
print("AUDIT DE QUALITÉ - GRANTS")
print("=" * 60)

# Valeurs manquantes
print("\nValeurs manquantes :")
print(grants.isnull().sum()[grants.isnull().sum() > 0])

# Doublons
print("\nNombre de doublons :", grants.duplicated().sum())

# Aperçu
print("\nAperçu des données :")
print(grants.head(2))


print("\n" + "=" * 60)
print("AUDIT DE QUALITÉ - NONPROFITS")
print("=" * 60)

# Valeurs manquantes
print("\nValeurs manquantes :")
print(nonprofits.isnull().sum()[nonprofits.isnull().sum() > 0])

# Doublons
print("\nNombre de doublons :", nonprofits.duplicated().sum())

# Aperçu
print("\nAperçu des données :")
print(nonprofits.head(2))


print("\n" + "=" * 60)
print("AUDIT DE QUALITÉ - QUALITY")
print("=" * 60)

# Valeurs manquantes
print("\nValeurs manquantes :")
print(quality.isnull().sum()[quality.isnull().sum() > 0])

# Doublons
print("\nNombre de doublons :", quality.duplicated().sum())

# Aperçu
print("\nAperçu des données :")
print(quality.head(2))

AUDIT DE QUALITÉ - GRANTS

Valeurs manquantes :
opportunity_number                       5978
opportunity_category                     5978
funding_instrument_type                  5978
category_of_funding_activity             5978
cfda_numbers                             6722
eligible_applicants                      5978
eligible_applicants_type                 5978
agency_code                              6014
agency_name                              6015
post_date                                5978
close_date                               9686
last_updated_date                        5978
archive_date                            12421
award_ceiling                           19149
award_floor                             26949
estimated_total_program_funding         26730
expected_number_of_awards               26586
cost_sharing_or_matching_requirement     5978
additional_information_url              28408
dtype: int64

Nombre de doublons : 0

Aperçu des données :
   Unnamed: 0  oppo

**4. Nettoyage :**

    * ✂️ Supprimer les 5,978 lignes vides de GRANTS
    * 🗑️ Enlever les colonnes inutiles (Unnamed)
    * 🔄 Convertir les dates en datetime
    * 🔗 Fusionner les datasets :
        * GRANTS + NONPROFITS via une clé commune (agence → organisation)
        * QUALITY pour enrichir les données de qualité

In [10]:
# === GRANTS ===
print("Nettoyage GRANTS...")

# Supprimer la colonne d'index
grants = grants.drop(columns=["Unnamed: 0"])

# Convertir les colonnes de date
date_columns = ["post_date", "close_date", "last_updated_date", "archive_date"]

for col in date_columns:
    grants[col] = pd.to_datetime(
        grants[col],
        format="%d/%m/%Y",
        errors="coerce"
    )

print(f"GRANTS : {grants.shape[0]:,} lignes × {grants.shape[1]} colonnes")


# === QUALITY ===
print("\nNettoyage QUALITY...")

# Supprimer la colonne presque vide
quality = quality.drop(columns=["Unnamed: 9"])

# Mettre EIN au même format que dans NONPROFITS
quality["EIN"] = quality["EIN"].astype(str)

print(f"QUALITY : {quality.shape[0]:,} lignes × {quality.shape[1]} colonnes")


# === NONPROFITS ===
print("\nPréparation NONPROFITS...")

# Mettre EIN au même format que dans QUALITY
nonprofits["EIN"] = nonprofits["EIN"].astype(str)

print(f"NONPROFITS : {nonprofits.shape[0]:,} lignes × {nonprofits.shape[1]} colonnes")


print("\nNettoyage terminé.")

Nettoyage GRANTS...
GRANTS : 75,337 lignes × 21 colonnes

Nettoyage QUALITY...
QUALITY : 240,585 lignes × 9 colonnes

Préparation NONPROFITS...
NONPROFITS : 240,585 lignes × 33 colonnes

Nettoyage terminé.


In [11]:
# Vérification après nettoyage

print("GRANTS")
print(grants.dtypes)

print("\n" + "-" * 60)

print("NONPROFITS")
print(nonprofits.dtypes)

print("\n" + "-" * 60)

print("QUALITY")
print(quality.dtypes)

GRANTS
opportunity_id                                   int64
opportunity_title                               object
opportunity_number                              object
opportunity_category                            object
funding_instrument_type                         object
category_of_funding_activity                    object
cfda_numbers                                   float64
eligible_applicants                             object
eligible_applicants_type                        object
agency_code                                     object
agency_name                                     object
post_date                               datetime64[ns]
close_date                              datetime64[ns]
last_updated_date                       datetime64[ns]
archive_date                            datetime64[ns]
award_ceiling                                  float64
award_floor                                    float64
estimated_total_program_funding                float64
exp

In [12]:
# Vérification de la clé EIN
print("NONPROFITS")
print("EIN uniques :", nonprofits["EIN"].nunique())
print("EIN en doublon :", nonprofits["EIN"].duplicated().sum())

print("\n" + "-" * 60)

print("QUALITY")
print("EIN uniques :", quality["EIN"].nunique())
print("EIN en doublon :", quality["EIN"].duplicated().sum())

NONPROFITS
EIN uniques : 240585
EIN en doublon : 0

------------------------------------------------------------
QUALITY
EIN uniques : 240578
EIN en doublon : 7


In [13]:
# Identifier les EIN en doublon dans QUALITY

doublons_quality = quality[quality["EIN"].duplicated(keep=False)]

print(doublons_quality.sort_values("EIN"))

                                                    NAME  EIN  \
10580             NORTH COUNTRY HOSPITAL & HEALTH CENTER  INC   
15722   GREATER BOSTON ASSOCIATION FOR RETARDED CITIZENS  INC   
48091                QUEENS COUNTY MENTAL HEALTH SOCIETY  INC   
79434                                 ASPIRA OF NEW YORK  INC   
96825            NIAGARA FALLS FIRE DEPT MUTUAL AID ASSN  INC   
113003                                 SPORTS FOUNDATION  INC   
143709                             BRIDGE OF WESTBOROUGH  INC   
146916                AUXILIARY OF PORTER MEDICAL CENTER  INC   

        confidence_score        data_quality missing_fields  \
10580         30185556.0  0.9999999999999999      excellent   
15722         42173649.0  0.7000000000000001           good   
48091        111776036.0  0.8999999999999999      excellent   
79434        136204790.0  0.9999999999999999      excellent   
96825        166027316.0  0.8999999999999999      excellent   
113003       221860827.0  0.99999999

In [14]:
# Vérifier le nombre de fois où chaque EIN apparaît

print(quality["EIN"].value_counts().loc[lambda x: x > 1])

EIN
INC    8
Name: count, dtype: int64


In [15]:
# Vérifier les lignes où EIN vaut "INC"

print(quality[quality["EIN"] == "INC"])

                                                    NAME  EIN  \
10580             NORTH COUNTRY HOSPITAL & HEALTH CENTER  INC   
15722   GREATER BOSTON ASSOCIATION FOR RETARDED CITIZENS  INC   
48091                QUEENS COUNTY MENTAL HEALTH SOCIETY  INC   
79434                                 ASPIRA OF NEW YORK  INC   
96825            NIAGARA FALLS FIRE DEPT MUTUAL AID ASSN  INC   
113003                                 SPORTS FOUNDATION  INC   
143709                             BRIDGE OF WESTBOROUGH  INC   
146916                AUXILIARY OF PORTER MEDICAL CENTER  INC   

        confidence_score        data_quality missing_fields  \
10580         30185556.0  0.9999999999999999      excellent   
15722         42173649.0  0.7000000000000001           good   
48091        111776036.0  0.8999999999999999      excellent   
79434        136204790.0  0.9999999999999999      excellent   
96825        166027316.0  0.8999999999999999      excellent   
113003       221860827.0  0.99999999

In [16]:
# Vérifier la structure des lignes concernées

print(quality.loc[quality["EIN"] == "INC", ["NAME", "EIN"]])

                                                    NAME  EIN
10580             NORTH COUNTRY HOSPITAL & HEALTH CENTER  INC
15722   GREATER BOSTON ASSOCIATION FOR RETARDED CITIZENS  INC
48091                QUEENS COUNTY MENTAL HEALTH SOCIETY  INC
79434                                 ASPIRA OF NEW YORK  INC
96825            NIAGARA FALLS FIRE DEPT MUTUAL AID ASSN  INC
113003                                 SPORTS FOUNDATION  INC
143709                             BRIDGE OF WESTBOROUGH  INC
146916                AUXILIARY OF PORTER MEDICAL CENTER  INC


In [17]:
# Rechercher les mêmes organisations dans NONPROFITS

noms = quality.loc[quality["EIN"] == "INC", "NAME"]

print(nonprofits[nonprofits["NAME"].isin(noms)][["NAME", "EIN"]])

Empty DataFrame
Columns: [NAME, EIN]
Index: []


In [18]:
# Vérifier les lignes problématiques dans QUALITY

print(quality.loc[quality["EIN"] == "INC"].to_string())

                                                    NAME  EIN  confidence_score        data_quality missing_fields         has_mission  has_financial  has_impact  has_basic
10580             NORTH COUNTRY HOSPITAL & HEALTH CENTER  INC        30185556.0  0.9999999999999999      excellent               set()           True        True       True
15722   GREATER BOSTON ASSOCIATION FOR RETARDED CITIZENS  INC        42173649.0  0.7000000000000001           good  {'financial_data'}           True       False       True
48091                QUEENS COUNTY MENTAL HEALTH SOCIETY  INC       111776036.0  0.8999999999999999      excellent      {'basic_info'}           True        True       True
79434                                 ASPIRA OF NEW YORK  INC       136204790.0  0.9999999999999999      excellent               set()           True        True       True
96825            NIAGARA FALLS FIRE DEPT MUTUAL AID ASSN  INC       166027316.0  0.8999999999999999      excellent      {'basic_info'} 

In [19]:
# Vérifier les noms avec leur représentation exacte

for nom in quality.loc[quality["EIN"] == "INC", "NAME"]:
    print(repr(nom))

'NORTH COUNTRY HOSPITAL & HEALTH CENTER'
'GREATER BOSTON ASSOCIATION FOR RETARDED CITIZENS'
'QUEENS COUNTY MENTAL HEALTH SOCIETY'
'ASPIRA OF NEW YORK'
'NIAGARA FALLS FIRE DEPT MUTUAL AID ASSN'
'SPORTS FOUNDATION'
'BRIDGE OF WESTBOROUGH'
'AUXILIARY OF PORTER MEDICAL CENTER'


In [20]:
# Vérifier les lignes problématiques dans le fichier QUALITY

quality_brut = pd.read_csv(
    "nonprofit_quality.csv",
    sep=";",
    encoding="latin1"
)

print(quality_brut.loc[10580:10582].to_string())

                                                NAME       EIN  confidence_score        data_quality                    missing_fields has_mission  has_financial  has_impact  has_basic Unnamed: 9
10580         NORTH COUNTRY HOSPITAL & HEALTH CENTER       INC        30185556.0  0.9999999999999999                         excellent       set()           True        True       True       True
10581  CHITTENDEN COUNTY FARM BUREAU ASSOCIATION INC  30185557               0.6                good  {'financial_data', 'basic_info'}        True          False        True      False        NaN
10582                      MORRISTOWN CEMETERY ASSOC  30185687               0.7                good                {'financial_data'}        True          False        True       True        NaN


C:\Users\PC\AppData\Local\Temp\ipykernel_18972\610266060.py:3: DtypeWarning: Columns (1,5,9) have mixed types. Specify dtype option on import or set low_memory=False.
  quality_brut = pd.read_csv(


In [21]:
# Vérifier toutes les lignes où EIN vaut INC

lignes_inc = quality_brut[quality_brut["EIN"] == "INC"]

print(lignes_inc[["NAME", "EIN", "confidence_score", "data_quality"]])

                                                    NAME  EIN  \
10580             NORTH COUNTRY HOSPITAL & HEALTH CENTER  INC   
15722   GREATER BOSTON ASSOCIATION FOR RETARDED CITIZENS  INC   
48091                QUEENS COUNTY MENTAL HEALTH SOCIETY  INC   
79434                                 ASPIRA OF NEW YORK  INC   
96825            NIAGARA FALLS FIRE DEPT MUTUAL AID ASSN  INC   
113003                                 SPORTS FOUNDATION  INC   
143709                             BRIDGE OF WESTBOROUGH  INC   
146916                AUXILIARY OF PORTER MEDICAL CENTER  INC   

        confidence_score        data_quality  
10580         30185556.0  0.9999999999999999  
15722         42173649.0  0.7000000000000001  
48091        111776036.0  0.8999999999999999  
79434        136204790.0  0.9999999999999999  
96825        166027316.0  0.8999999999999999  
113003       221860827.0  0.9999999999999999  
143709       237203001.0  0.9999999999999999  
146916       237363227.0  0.9999999999

In [22]:
# Vérifier les correspondances entre NONPROFITS et QUALITY

ein_nonprofits = set(nonprofits["EIN"])
ein_quality = set(quality["EIN"])

correspondances = len(ein_nonprofits.intersection(ein_quality))

print("EIN correspondants :", correspondances)
print("EIN dans NONPROFITS :", len(ein_nonprofits))
print("EIN dans QUALITY :", len(ein_quality))

EIN correspondants : 240577
EIN dans NONPROFITS : 240585
EIN dans QUALITY : 240578


In [23]:
# Fusion des données NONPROFITS et QUALITY

nonprofits_complet = nonprofits.merge(
    quality,
    on="EIN",
    how="left"
)

print("Fusion terminée.")
print(f"Nombre de lignes : {len(nonprofits_complet):,}")
print(f"Nombre de colonnes : {nonprofits_complet.shape[1]}")

Fusion terminée.
Nombre de lignes : 240,585
Nombre de colonnes : 41


In [24]:
# Vérifier les résultats de la fusion

print("Organisations avec une information de qualité :")
print(nonprofits_complet["data_quality"].notna().sum())

print("\nOrganisations sans information de qualité :")
print(nonprofits_complet["data_quality"].isna().sum())

Organisations avec une information de qualité :
240577

Organisations sans information de qualité :
8


In [25]:
# Vérifier les colonnes après la fusion

print("Colonnes disponibles :")
print(nonprofits_complet.columns.tolist())

Colonnes disponibles :
['EIN', 'NAME_x', 'ICO', 'STREET', 'CITY', 'STATE', 'ZIP', 'GROUP', 'SUBSECTION', 'AFFILIATION', 'CLASSIFICATION', 'RULING', 'DEDUCTIBILITY', 'FOUNDATION', 'ACTIVITY', 'ORGANIZATION', 'STATUS', 'TAX_PERIOD', 'ASSET_CD', 'INCOME_CD', 'FILING_REQ_CD', 'PF_FILING_REQ_CD', 'ACCT_PD', 'ASSET_AMT', 'INCOME_AMT', 'REVENUE_AMT', 'NTEE_CD', 'SORT_NAME', 'impact_score', 'mission_statement', 'financial_metric', 'impact_score_numeric', 'impact_efficiency', 'NAME_y', 'confidence_score', 'data_quality', 'missing_fields', 'has_mission', 'has_financial', 'has_impact', 'has_basic']


In [26]:
# Examiner les principales variables numériques

print("Statistiques des variables financières et d'impact :")

print(nonprofits_complet[
    [
        "ASSET_AMT",
        "INCOME_AMT",
        "REVENUE_AMT",
        "financial_metric",
        "impact_score_numeric",
        "impact_efficiency",
        "confidence_score"
    ]
].describe())

Statistiques des variables financières et d'impact :
          ASSET_AMT    INCOME_AMT   REVENUE_AMT  financial_metric  \
count  1.962130e+05  1.962140e+05  2.405850e+05      2.405850e+05   
mean   6.278859e+06  4.438425e+06  2.267515e+06      2.268317e+06   
std    2.324241e+08  2.316582e+08  8.563901e+07      8.563893e+07   
min    0.000000e+00 -1.837054e+06 -3.828561e+07      1.000000e+00   
25%    0.000000e+00  0.000000e+00  0.000000e+00      1.000000e+00   
50%    1.736100e+04  2.235150e+04  0.000000e+00      1.000000e+00   
75%    3.731420e+05  2.254808e+05  6.664900e+04      6.664900e+04   
max    7.528751e+10  8.533663e+10  3.277372e+10      3.277372e+10   

       impact_score_numeric  confidence_score  
count         240585.000000     240577.000000  
mean               1.287765          0.806430  
std                0.594184          0.161404  
min                1.000000          0.600000  
25%                1.000000          0.700000  
50%                1.000000          

In [27]:
# Vérifier les valeurs négatives dans les variables financières

print("INCOME_AMT négatif :", (nonprofits_complet["INCOME_AMT"] < 0).sum())
print("REVENUE_AMT négatif :", (nonprofits_complet["REVENUE_AMT"] < 0).sum())

print("\nValeurs négatives de INCOME_AMT :")
print(nonprofits_complet.loc[
    nonprofits_complet["INCOME_AMT"] < 0,
    ["NAME_x", "INCOME_AMT"]
].head())

print("\nValeurs négatives de REVENUE_AMT :")
print(nonprofits_complet.loc[
    nonprofits_complet["REVENUE_AMT"] < 0,
    ["NAME_x", "REVENUE_AMT"]
].head())

INCOME_AMT négatif : 44
REVENUE_AMT négatif : 461

Valeurs négatives de INCOME_AMT :
                                                  NAME_x  INCOME_AMT
8760                           THE HIGH Q FOUNDATION INC    -60426.0
13484  AUGUSTUS & KATHLEEN BARROWS MEMORIAL AND TRUST...     -7656.0
28280    JEWISH COMMUNITY HOUSING FOR THE ELDERLY VI INC  -1837054.0
28810                                STAEDEL FRIENDS INC      -353.0
29429                                 UBS FOUNDATION USA     -6698.0

Valeurs négatives de REVENUE_AMT :
                                                 NAME_x  REVENUE_AMT
494                  NATIONAL SOCIETY OF COLONIAL DAMES     -24169.0
642   VETERANS OF FOREIGN WARS OF THE UNITED STATES ...      -7332.0
673                                 OCEANVIEW MANOR INC    -103231.0
1335                   ALLAGASH DEVELOPMENT CORPORATION       -655.0
1996                   ATLANTIC VOLUNTEER ENGINE CO INC      -3200.0


In [28]:
# Vérifier les valeurs extrêmes de REVENUE_AMT

print(
    nonprofits_complet[
        ["NAME_x", "REVENUE_AMT"]
    ]
    .sort_values("REVENUE_AMT", ascending=False)
    .head(10)
)

                                                   NAME_x   REVENUE_AMT
172333          SELENA THOMAS JACKSON PROFIT ORGANIZATION  3.277372e+10
237668                     PARTNERS HEALTHCARE SYSTEM INC  1.111736e+10
54885             NEW YORK STATE CATHOLIC HEALTH PLAN INC  6.493574e+09
77059                                 NEW YORK UNIVERSITY  6.043438e+09
63237           HEALTH INSURANCE PLAN OF GREATER NEW YORK  5.161648e+09
71721                                HEALTHFIRST PHSP INC  5.133154e+09
73536                  NEW YORK AND PRESBYTERIAN HOSPITAL  4.816088e+09
36380                                     YALE UNIVERSITY  4.807998e+09
47426           FIDELITY INVESTMENTS CHARITABLE GIFT FUND  4.780943e+09
77230   TRUSTEES OF COLUMBIA UNIVERSITY IN THE CITY OF...  4.708226e+09


In [29]:
# Vérifier les valeurs extrêmes de ASSET_AMT

print(
    nonprofits_complet[
        ["NAME_x", "ASSET_AMT"]
    ]
    .sort_values("ASSET_AMT", ascending=False)
    .head(10)
)

                                                   NAME_x     ASSET_AMT
14718            PRESIDENT AND FELLOWS OF HARVARD COLLEGE  7.528751e+10
36380                                     YALE UNIVERSITY  3.410300e+10
35921                                 KNIGHTS OF COLUMBUS  2.222452e+10
14728               MASSACHUSETTS INSTITUTE OF TECHNOLOGY  2.217703e+10
77230   TRUSTEES OF COLUMBIA UNIVERSITY IN THE CITY OF...  1.667260e+10
47426           FIDELITY INVESTMENTS CHARITABLE GIFT FUND  1.609208e+10
237668                     PARTNERS HEALTHCARE SYSTEM INC  1.498361e+10
21569       HARVARD MANAGEMENT PRIVATE EQUITY CORPORATION  1.439566e+10
87069                                  CORNELL UNIVERSITY  1.236990e+10
62795                                     FORD FOUNDATION  1.211400e+10


In [30]:
# Vérifier l'organisation avec le revenu le plus élevé

print(
    nonprofits_complet.loc[
        nonprofits_complet["REVENUE_AMT"].idxmax()
    ]
)

EIN                                                             311685372
NAME_x                          SELENA THOMAS JACKSON PROFIT ORGANIZATION
ICO                                             % MT OLIVE UNITED MISSION
STREET                                                    39 COTTAGE ST 2
CITY                                                                 LYNN
STATE                                                                  MA
ZIP                                                            01905-2306
GROUP                                                                 0.0
SUBSECTION                                                             91
AFFILIATION                                                           0.0
CLASSIFICATION                                                     1000.0
RULING                                                                0.0
DEDUCTIBILITY                                                         0.0
FOUNDATION                            

In [31]:
# Répartition des niveaux d'impact

print("Répartition de l'impact :")
print(nonprofits_complet["impact_score"].value_counts(dropna=False))

Répartition de l'impact :
impact_score
Low       189168
Medium     33602
High       17815
Name: count, dtype: int64


In [32]:
# Répartition de la qualité des données

print("\nQualité des données :")
print(nonprofits_complet["data_quality"].value_counts(dropna=False))


Qualité des données :
data_quality
good         125733
excellent    114844
NaN               8
Name: count, dtype: int64


In [ ]:
# vérifier si les variables d'impact sont cohérentes entre elles
print("Valeurs de impact_score_numeric :")
print(nonprofits_complet["impact_score_numeric"].value_counts().sort_index())

print("\nValeurs de impact_efficiency :")
print(nonprofits_complet["impact_efficiency"].value_counts(dropna=False))

Valeurs de impact_score_numeric :
impact_score_numeric
1    189168
2     33602
3     17815
Name: count, dtype: int64

Valeurs de impact_efficiency :
impact_efficiency
1.0                       140972
1.0                        10040
0.002                         40
0.01                          40
0.0002                        34
                           ...  
1.042265969599186e-06          1
6.215920181617585e-08          1
1.211108285191779e-05          1
5.976804023584469e-06          1
2.9957251002818976e-06         1
Name: count, Length: 79739, dtype: int64


In [34]:
# Vérifier le type, les valeurs manquantes et le contenu de la variable impact_efficiency.

print("Type de impact_efficiency :", nonprofits_complet["impact_efficiency"].dtype)

print("\nNombre de valeurs manquantes :")
print(nonprofits_complet["impact_efficiency"].isna().sum())

print("\nQuelques valeurs :")
print(nonprofits_complet["impact_efficiency"].head(20).to_list())

Type de impact_efficiency : object

Nombre de valeurs manquantes :
0

Quelques valeurs :
['1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.0', '1.8182148766341205e-05', '1.9898517560441747e-05', '1.0', '1.327509989512671e-05']


In [37]:
# Convertir impact_efficiency en valeur numérique.
nonprofits_complet["impact_efficiency"] = pd.to_numeric(
    nonprofits_complet["impact_efficiency"],
    errors="coerce"
)

# Vérifier la conversion de impact_efficiency.
print(nonprofits_complet["impact_efficiency"].dtype)

float64


In [ ]:
# Afficher les statistiques descriptives de impact_efficiency
print(nonprofits_complet["impact_efficiency"].describe())

count    2.404970e+05
mean     6.281461e-01
std      4.831235e-01
min      9.153676e-11
25%      1.727772e-05
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: impact_efficiency, dtype: float64


In [39]:
# Comparer l'efficacité moyenne selon le niveau d'impact.
print(
    nonprofits_complet.groupby("impact_score")["impact_efficiency"].mean()
)

impact_score
High      0.000001
Low       0.798886
Medium    0.000009
Name: impact_efficiency, dtype: float64


In [40]:
# Vérifier l'efficacité par niveau d'impact.
print(
    nonprofits_complet.groupby("impact_score")["impact_efficiency"].describe()
)

                 count      mean           std           min           25%  \
impact_score                                                                 
High           17813.0  0.000001  8.768404e-07  9.153676e-11  2.875673e-07   
Low           189097.0  0.798886  4.005646e-01  1.000020e-05  1.000000e+00   
Medium         33587.0  0.000009  5.186672e-06  2.000246e-06  4.379137e-06   

                       50%       75%       max  
impact_score                                    
High          8.362465e-07  0.000002  0.000003  
Low           1.000000e+00  1.000000  1.000000  
Medium        8.066826e-06  0.000013  0.000020  


In [41]:
# Vérifier la relation entre l'efficacité et la performance financière.
print(
    nonprofits_complet[
        ["impact_efficiency", "financial_metric"]
    ].corr()
)

                   impact_efficiency  financial_metric
impact_efficiency           1.000000         -0.034437
financial_metric           -0.034437          1.000000


In [43]:
# Vérifier le type des indicateurs de qualité.
print(nonprofits_complet[
    ["has_mission", "has_financial", "has_impact", "has_basic"]
].dtypes)

has_mission      object
has_financial    object
has_impact       object
has_basic        object
dtype: object


In [45]:
# Convertir les indicateurs de qualité en valeurs booléennes.
colonnes_qualite = ["has_mission", "has_financial", "has_impact", "has_basic"]

for colonne in colonnes_qualite:
    nonprofits_complet[colonne] = nonprofits_complet[colonne].astype(bool)
    
print(nonprofits_complet[colonnes_qualite].dtypes)

has_mission      bool
has_financial    bool
has_impact       bool
has_basic        bool
dtype: object


In [46]:
# Compter les organisations disposant de chaque information.
print(nonprofits_complet[colonnes_qualite].sum())

has_mission      240585
has_financial    114852
has_impact       240585
has_basic        152100
dtype: int64


In [47]:
# Calculer le taux de disponibilité de chaque information.
print(
    (nonprofits_complet[colonnes_qualite].mean() * 100).round(2)
)

has_mission      100.00
has_financial     47.74
has_impact       100.00
has_basic         63.22
dtype: float64


In [48]:
# Afficher les colonnes disponibles après le nettoyage et la fusion.
print(nonprofits_complet.columns.tolist())

['EIN', 'NAME_x', 'ICO', 'STREET', 'CITY', 'STATE', 'ZIP', 'GROUP', 'SUBSECTION', 'AFFILIATION', 'CLASSIFICATION', 'RULING', 'DEDUCTIBILITY', 'FOUNDATION', 'ACTIVITY', 'ORGANIZATION', 'STATUS', 'TAX_PERIOD', 'ASSET_CD', 'INCOME_CD', 'FILING_REQ_CD', 'PF_FILING_REQ_CD', 'ACCT_PD', 'ASSET_AMT', 'INCOME_AMT', 'REVENUE_AMT', 'NTEE_CD', 'SORT_NAME', 'impact_score', 'mission_statement', 'financial_metric', 'impact_score_numeric', 'impact_efficiency', 'NAME_y', 'confidence_score', 'data_quality', 'missing_fields', 'has_mission', 'has_financial', 'has_impact', 'has_basic']


In [49]:
# Sélectionner les variables utiles pour l'analyse.
colonnes_analyse = [
    "EIN",
    "NAME_x",
    "CITY",
    "STATE",
    "ASSET_AMT",
    "INCOME_AMT",
    "REVENUE_AMT",
    "financial_metric",
    "impact_score",
    "impact_score_numeric",
    "confidence_score",
    "data_quality",
    "missing_fields",
    "has_mission",
    "has_financial",
    "has_impact",
    "has_basic"
]

organisations = nonprofits_complet[colonnes_analyse].copy()

print("Table analytique :", organisations.shape)

Table analytique : (240585, 17)


In [50]:
# Vérifier l'unicité de l'identifiant EIN.
print("Nombre d'EIN uniques :", organisations["EIN"].nunique())
print("Nombre de doublons :", organisations["EIN"].duplicated().sum())

Nombre d'EIN uniques : 240585
Nombre de doublons : 0


In [52]:
# Renommer les colonnes en français.
organisations = organisations.rename(columns={
    "EIN": "identifiant",
    "NAME": "nom_organisation",
    "CITY": "ville",
    "STATE": "etat",
    "ASSET_AMT": "montant_actifs",
    "INCOME_AMT": "montant_revenus",
    "REVENUE_AMT": "chiffre_affaires",
    "financial_metric": "indicateur_financier",
    "impact_score": "niveau_impact",
    "impact_score_numeric": "score_impact",
    "confidence_score": "score_confiance",
    "data_quality": "qualite_donnees",
    "missing_fields": "champs_manquants",
    "has_mission": "mission_disponible",
    "has_financial": "donnees_financieres_disponibles",
    "has_impact": "impact_disponible",
    "has_basic": "informations_base_disponibles"
})

print(organisations.columns.tolist())

['identifiant', 'nom_organisation', 'ville', 'etat', 'montant_actifs', 'montant_revenus', 'chiffre_affaires', 'indicateur_financier', 'niveau_impact', 'score_impact', 'score_confiance', 'qualite_donnees', 'champs_manquants', 'mission_disponible', 'donnees_financieres_disponibles', 'impact_disponible', 'informations_base_disponibles']


In [53]:
# Vérifier les valeurs manquantes dans la table analytique.
print(organisations.isnull().sum()[organisations.isnull().sum() > 0])

ville                   1
etat                    1
montant_actifs      44372
montant_revenus     44371
score_confiance         8
qualite_donnees         8
champs_manquants        8
dtype: int64


In [54]:
# Vérifier la cohérence des données financières manquantes.
print(
    organisations.groupby("donnees_financieres_disponibles")[
        ["montant_actifs", "montant_revenus"]
    ].apply(lambda x: x.isnull().sum())
)

                                 montant_actifs  montant_revenus
donnees_financieres_disponibles                                 
False                                     44371            44371
True                                          1                0


In [55]:
# Vérifier les valeurs de champs_manquants.
print(organisations["champs_manquants"].value_counts(dropna=False).head(10))

champs_manquants
set()                               77704
{'financial_data'}                  74388
{'financial_data', 'basic_info'}    51345
{'basic_info'}                      37140
NaN                                     8
Name: count, dtype: int64


In [56]:
# Vérifier la répartition des organisations selon les informations disponibles.
print(
    organisations[
        [
            "mission_disponible",
            "donnees_financieres_disponibles",
            "impact_disponible",
            "informations_base_disponibles"
        ]
    ].value_counts()
)

mission_disponible  donnees_financieres_disponibles  impact_disponible  informations_base_disponibles
True                True                             True               True                             77712
                    False                            True               True                             74388
                                                                        False                            51345
                    True                             True               False                            37140
Name: count, dtype: int64


In [58]:
# Identifier les organisations sans information de qualité.
print(
    organisations[
        organisations["score_confiance"].isna()
    ][
        ["identifiant", "nom_organisation", "score_confiance",
         "qualite_donnees", "champs_manquants"]
    ]
)

       identifiant                                   nom_organisation  \
10580     30185556         NORTH COUNTRY HOSPITAL & HEALTH CENTER,INC   
15722     42173649  GREATER BOSTON ASSOCIATION FOR RETARDED CITIZE...   
48091    111776036            QUEENS COUNTY MENTAL HEALTH SOCIETY,INC   
79434    136204790                             ASPIRA OF NEW YORK,INC   
96825    166027316        NIAGARA FALLS FIRE DEPT MUTUAL AID ASSN,INC   
113003   221860827                              SPORTS FOUNDATION,INC   
143709   237203001                          BRIDGE OF WESTBOROUGH,INC   
146916   237363227             AUXILIARY OF PORTER MEDICAL CENTER,INC   

        score_confiance qualite_donnees champs_manquants  
10580               NaN             NaN              NaN  
15722               NaN             NaN              NaN  
48091               NaN             NaN              NaN  
79434               NaN             NaN              NaN  
96825               NaN             NaN        

In [59]:
# Vérifier les valeurs manquantes restantes.
print(organisations.isnull().sum()[organisations.isnull().sum() > 0])

ville                   1
etat                    1
montant_actifs      44372
montant_revenus     44371
score_confiance         8
qualite_donnees         8
champs_manquants        8
dtype: int64


In [60]:
# Vérifier les types de données de la table finale.
print(organisations.dtypes)

identifiant                         object
nom_organisation                    object
ville                               object
etat                                object
montant_actifs                     float64
montant_revenus                    float64
chiffre_affaires                   float64
indicateur_financier               float64
niveau_impact                       object
score_impact                         int64
score_confiance                    float64
qualite_donnees                     object
champs_manquants                    object
mission_disponible                    bool
donnees_financieres_disponibles       bool
impact_disponible                     bool
informations_base_disponibles         bool
dtype: object


In [61]:
# Vérifier les types de données de la table grants.
print(grants.dtypes)

opportunity_id                                   int64
opportunity_title                               object
opportunity_number                              object
opportunity_category                            object
funding_instrument_type                         object
category_of_funding_activity                    object
cfda_numbers                                   float64
eligible_applicants                             object
eligible_applicants_type                        object
agency_code                                     object
agency_name                                     object
post_date                               datetime64[ns]
close_date                              datetime64[ns]
last_updated_date                       datetime64[ns]
archive_date                            datetime64[ns]
award_ceiling                                  float64
award_floor                                    float64
estimated_total_program_funding                float64
expected_n

In [62]:
# Vérifier le contenu de cfda_numbers.
print(grants["cfda_numbers"].head(20).to_list())

[19.04, nan, 15.945, 15.945, 19.415, 19.009, 15.954, 15.945, 15.945, 15.945, 15.945, 15.945, 15.945, 15.944, 15.945, 15.655, 15.944, 15.954, 98.001, 15.939]


In [64]:
# Convertir cfda_numbers en texte.
grants["cfda_numbers"] = grants["cfda_numbers"].astype("string")
print(grants["cfda_numbers"].dtype)

string


In [65]:
# Vérifier les valeurs négatives des montants financiers.
print("award_ceiling négatif :", (grants["award_ceiling"] < 0).sum())
print("award_floor négatif :", (grants["award_floor"] < 0).sum())
print("estimated_total_program_funding négatif :", (grants["estimated_total_program_funding"] < 0).sum())

award_ceiling négatif : 0
award_floor négatif : 0
estimated_total_program_funding négatif : 0


In [66]:
# Vérifier la cohérence entre le montant minimum et le montant maximum.
print(
    "award_floor supérieur à award_ceiling :",
    (grants["award_floor"] > grants["award_ceiling"]).sum()
)

award_floor supérieur à award_ceiling : 1


In [67]:
# Identifier la ligne où le montant minimum dépasse le montant maximum.
print(
    grants.loc[
        grants["award_floor"] > grants["award_ceiling"],
        [
            "opportunity_id",
            "opportunity_title",
            "award_floor",
            "award_ceiling"
        ]
    ]
)

       opportunity_id                                  opportunity_title  \
20076          284822  Comprehensive High-Impact HIV Prevention Proje...   

       award_floor  award_ceiling  
20076     325000.0            0.0  


In [68]:
# Signaler les montants incohérents sans modifier les valeurs originales.
grants["montants_incoherents"] = (
    grants["award_floor"] > grants["award_ceiling"]
)

print("Lignes avec des montants incohérents :",
      grants["montants_incoherents"].sum())

Lignes avec des montants incohérents : 1


In [69]:
# Vérifier les valeurs manquantes des variables financières.
print(
    grants[
        [
            "award_ceiling",
            "award_floor",
            "estimated_total_program_funding",
            "expected_number_of_awards"
        ]
    ].isnull().sum()
)

award_ceiling                      19149
award_floor                        26949
estimated_total_program_funding    26730
expected_number_of_awards          26586
dtype: int64


In [70]:
# Vérifier les combinaisons de valeurs manquantes des variables financières.
print(
    grants[
        [
            "award_ceiling",
            "award_floor",
            "estimated_total_program_funding",
            "expected_number_of_awards"
        ]
    ].isnull().value_counts()
)

award_ceiling  award_floor  estimated_total_program_funding  expected_number_of_awards
False          False        False                            False                        36291
True           True         True                             True                         13452
False          False        True                             False                         4774
                                                             True                          4122
               True         True                             True                          3545
True           True         False                            False                         3512
False          True         False                            False                         3138
               False        False                            True                          2886
True           True         False                            True                          1329
False          True         False                

In [71]:
# Vérifier la cohérence entre les dates de publication et de clôture.
print(
    "Dates de clôture antérieures aux dates de publication :",
    (grants["close_date"] < grants["post_date"]).sum()
)

Dates de clôture antérieures aux dates de publication : 0


In [72]:
# Vérifier les valeurs manquantes des dates.
print(grants[
    ["post_date", "close_date", "last_updated_date", "archive_date"]
].isnull().sum())

post_date             5978
close_date            9686
last_updated_date     5978
archive_date         12421
dtype: int64


In [73]:
# Vérifier la cohérence entre les dates de publication et d'archivage.
print(
    "Dates d'archivage antérieures aux dates de publication :",
    (grants["archive_date"] < grants["post_date"]).sum()
)

Dates d'archivage antérieures aux dates de publication : 0


In [74]:
# Vérifier les catégories d'opportunités.
print(grants["opportunity_category"].value_counts(dropna=False))

opportunity_category
Discretionary    64752
NaN               5978
Other             2025
Mandatory         1135
Continuation      1049
Earmark            398
Name: count, dtype: int64


In [75]:
# Vérifier les types d'instruments de financement.
print(grants["funding_instrument_type"].value_counts(dropna=False))

funding_instrument_type
Cooperative Agreement    33531
Grant                    33155
NaN                       5978
Other                     1711
Procurement Contract       962
Name: count, dtype: int64


In [76]:
# Vérifier les catégories d'activité de financement.
print(grants["category_of_funding_activity"].value_counts(dropna=False))

category_of_funding_activity
Health                                                       15412
Science and Technology and other Research and Development    11629
Natural Resources                                            10746
Other                                                         8386
NaN                                                           5978
Income Security and Social Services                           4474
Education                                                     3739
Environment                                                   3496
Law, Justice and Legal Services                               2137
Humanities                                                    1152
Agriculture                                                   1134
Energy                                                        1031
Transportation                                                1028
Employment, Labor and Training                                 915
Community Development            

In [77]:
# Vérifier les catégories de bénéficiaires éligibles.
print(grants["eligible_applicants"].value_counts(dropna=False).head(20))

eligible_applicants
Others                                                                                                        39809
Unrestricted                                                                                                   9896
NaN                                                                                                            5978
Nonprofits having a 501 (c) (3) status with the IRS, other than institutions of higher education               4433
Private institutions of higher education                                                                       3461
Small businesses                                                                                               3401
Public and State controlled institutions of higher education                                                   2534
State governments                                                                                              2049
Nonprofits that do not have a 501 (c) (3) status wit

In [78]:
# Vérifier les types de bénéficiaires éligibles.
print(grants["eligible_applicants_type"].value_counts(dropna=False))

eligible_applicants_type
Non-Government Organization    53344
Any                             9896
Government                      6119
NaN                             5978
Name: count, dtype: int64


In [79]:
# Vérifier le contenu du code de l'agence.
print(grants["agency_code"].value_counts(dropna=False).head(20))

agency_code
HHS-NIH11    11067
DOI-NPS       7734
NaN           6014
DOI-FWS       3844
DOI-BLM       2768
DOI-USGS1     2258
HHS-HRSA      1448
DOD-AMRAA     1417
ED            1360
NASA-HQ       1319
EPA           1281
NSF           1076
HHS-CDC       1003
DHS-DHS        941
DOC            859
HHS-ACF        837
DOI-BOR        789
USDA-NIFA      737
USAID          734
DOE-GFO        672
Name: count, dtype: int64


In [80]:
# Vérifier les noms des agences.
print(grants["agency_name"].value_counts(dropna=False).head(20))

agency_name
National Institutes of Health                   11068
National Park Service                            7734
NaN                                              6015
Fish and Wildlife Service                        3844
Bureau of Land Management                        2768
Geological Survey                                2258
Health Resources and Services Administration     1448
Dept. of the Army -- USAMRAA                     1417
Department of Education                          1360
NASA Headquarters                                1319
Environmental Protection Agency                  1281
National Science Foundation                      1076
Centers for Disease Control and Prevention       1003
Department of Homeland Security - FEMA            941
Department of Commerce                            859
Administration for Children and Families          837
Bureau of Reclamation                             789
National Institute of Food and Agriculture        737
Agency for Inter

In [81]:
# Vérifier les exigences de financement complémentaire.
print(grants["cost_sharing_or_matching_requirement"].value_counts(dropna=False))

cost_sharing_or_matching_requirement
False    57921
True     11438
NaN       5978
Name: count, dtype: int64


In [82]:
# Convertir l'exigence de financement complémentaire en booléen.
grants["cost_sharing_or_matching_requirement"] = grants[
    "cost_sharing_or_matching_requirement"
].astype("boolean")

print(grants["cost_sharing_or_matching_requirement"].dtype)

boolean


In [83]:
# Vérifier le contenu des liens d'information complémentaires.
print(grants["additional_information_url"].head(10).to_list())

[nan, nan, nan, nan, 'http://exchanges.state.gov/grants/open2.html', 'http://exchanges.state.gov/grants/open2.html', nan, nan, nan, nan]


In [85]:
# Renommer les colonnes de grants en français.
grants = grants.rename(columns={
    "opportunity_id": "identifiant_opportunite",
    "opportunity_title": "titre_opportunite",
    "opportunity_number": "numero_opportunite",
    "opportunity_category": "categorie_opportunite",
    "funding_instrument_type": "type_financement",
    "category_of_funding_activity": "secteur_financement",
    "cfda_numbers": "numero_cfda",
    "eligible_applicants": "beneficiaires_eligibles",
    "eligible_applicants_type": "type_beneficiaire",
    "agency_code": "code_agence",
    "agency_name": "nom_agence",
    "post_date": "date_publication",
    "close_date": "date_cloture",
    "last_updated_date": "date_mise_a_jour",
    "archive_date": "date_archivage",
    "award_ceiling": "montant_maximal",
    "award_floor": "montant_minimal",
    "estimated_total_program_funding": "financement_total_estime",
    "expected_number_of_awards": "nombre_attributions_prevu",
    "cost_sharing_or_matching_requirement": "financement_complementaire_requis",
    "additional_information_url": "url_information"
})

print(grants.columns.tolist())

['identifiant_opportunite', 'titre_opportunite', 'numero_opportunite', 'categorie_opportunite', 'type_financement', 'secteur_financement', 'numero_cfda', 'beneficiaires_eligibles', 'type_beneficiaire', 'code_agence', 'nom_agence', 'date_publication', 'date_cloture', 'date_mise_a_jour', 'date_archivage', 'montant_maximal', 'montant_minimal', 'financement_total_estime', 'nombre_attributions_prevu', 'financement_complementaire_requis', 'url_information', 'montants_incoherents']


In [86]:
# Vérification finale de la table grants.
print("Dimensions :", grants.shape)
print("\nDoublons :", grants.duplicated().sum())
print("\nTypes de données :")
print(grants.dtypes)

Dimensions : (75337, 22)

Doublons : 0

Types de données :
identifiant_opportunite                       int64
titre_opportunite                            object
numero_opportunite                           object
categorie_opportunite                        object
type_financement                             object
secteur_financement                          object
numero_cfda                          string[python]
beneficiaires_eligibles                      object
type_beneficiaire                            object
code_agence                                  object
nom_agence                                   object
date_publication                     datetime64[ns]
date_cloture                         datetime64[ns]
date_mise_a_jour                     datetime64[ns]
date_archivage                       datetime64[ns]
montant_maximal                             float64
montant_minimal                             float64
financement_total_estime                    float64
nombr

In [87]:
# Vérifier les valeurs manquantes après le nettoyage.
print(grants.isnull().sum()[grants.isnull().sum() > 0])

numero_opportunite                    5978
categorie_opportunite                 5978
type_financement                      5978
secteur_financement                   5978
numero_cfda                           6722
beneficiaires_eligibles               5978
type_beneficiaire                     5978
code_agence                           6014
nom_agence                            6015
date_publication                      5978
date_cloture                          9686
date_mise_a_jour                      5978
date_archivage                       12421
montant_maximal                      19149
montant_minimal                      26949
financement_total_estime             26730
nombre_attributions_prevu            26586
financement_complementaire_requis     5978
url_information                      28408
dtype: int64


In [88]:
# Vérifier les tables finales.
print("Organisations :", organisations.shape)
print("Grants :", grants.shape)

Organisations : (240585, 17)
Grants : (75337, 22)


In [89]:
# Exporter les données nettoyées pour Power BI

organisations.to_csv("organisations.csv", index=False, encoding="utf-8-sig")
grants.to_csv("subventions.csv", index=False, encoding="utf-8-sig")

print("Export terminé.")

Export terminé.
